# Example: Fixed-core divertor shaping with virtual circuits

---

This example demonstrates how existing FreeGSNKE **virtual circuit** functionality can be used to make a large change to the divertor magnetic geometry while keeping the core plasma boundary approximately fixed.

The workflow is inspired by the point of view used in O. P. Bardsley *et al.*, *The Tokamak Exhaust Designer: a new tool for enhanced divertor control on MAST-U* (2025). That method constrains a spherical-harmonic representation of the vacuum field over the core. Here we construct a simpler demonstration using only existing FreeGSNKE functionality:

- sample the coil contribution to poloidal flux, $\psi_{\mathrm{tokamak}}$, at fixed points on the initial core boundary;
- request zero change in $\psi_{\mathrm{tokamak}}$ at those points;
- request a change in $\psi_{\mathrm{tokamak}}$ at a point on the divertor target; and
- use a virtual circuit to find and apply the corresponding PF-coil current changes.

This is a demonstration of pointwise flux control, not a complete implementation of the paper's method. In particular, the virtual-circuit calculation used here does not enforce PF-current limits or optimise a weighted distance from the original currents.

### Generate the starting equilibrium

We first reproduce the initial diverted MAST-U-like equilibrium from Example 02. The machine description, grid, profile parameters, stored coil currents, and static forward solve settings are kept the same.

In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.distance import cdist

In [ ]:
# build the MAST-U-like machine
from freegsnke import build_machine

tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path="../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path="../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

In [ ]:
# initialise the equilibrium and plasma profiles
from freegsnke import equilibrium_update
from freegsnke.jtor_update import ConstrainPaxisIp

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1,
    Rmax=2.0,
    Zmin=-2.2,
    Zmax=2.2,
    nx=65,
    ny=129,
)

profiles = ConstrainPaxisIp(
    eq=eq,
    paxis=8e3,
    Ip=6e5,
    fvac=0.5,
    alpha_m=1.8,
    alpha_n=1.2,
)

In [ ]:
# load the nonlinear static solver and the stored PF-coil currents
from freegsnke import GSstaticsolver

GSStaticSolver = GSstaticsolver.NKGSsolver(eq, gs_operator_order=4)

with open("data/simple_diverted_currents_PaxisIp.pk", "rb") as f:
    currents_dict = pickle.load(f)

for coil, current in currents_dict.items():
    eq.tokamak.set_coil_current(coil, current)

GSStaticSolver.solve(
    eq=eq,
    profiles=profiles,
    constrain=None,
    target_relative_tolerance=1e-9,
    verbose=True,
)

# refresh the plasma-flux interpolator used by psiRZ below
eq._updatePlasmaPsi(eq.plasma_psi)

In [ ]:
# inspect the starting equilibrium
initial_strikepoints = eq.strikepoints()
print("Initial strike points [m]:")
print(initial_strikepoints)

fig, ax = plt.subplots(figsize=(4, 8), dpi=80)
ax.grid(True, which="both", alpha=0.5)
eq.plot(axis=ax, show=False)
eq.tokamak.plot(axis=ax, show=False)
ax.set_xlim(0.1, 2.15)
ax.set_ylim(-2.25, 2.25)
plt.tight_layout()

### Define the core and divertor control points

The core constraints are evaluated at 16 fixed points sampled from the initial separatrix. These coordinates remain fixed after the coil currents are changed: the target is to preserve the initial vacuum-field pattern in the core region, rather than to follow the subsequently calculated boundary.

The desired outer strike-point radius is $R=1.5$ m. We locate the lowest intersection of that radius with the lower wall and use it as a divertor flux-control point.

In [ ]:
def lower_wall_point_at_R(equilibrium, target_R):
    """Return the lowest point where the machine wall crosses target_R."""

    wall = np.column_stack(
        [equilibrium.tokamak.wall.R, equilibrium.tokamak.wall.Z]
    )
    intersections = []

    for point_1, point_2 in zip(wall, np.roll(wall, -1, axis=0)):
        R_1, Z_1 = point_1
        R_2, Z_2 = point_2

        if np.isclose(R_1, R_2):
            continue
        if min(R_1, R_2) <= target_R <= max(R_1, R_2):
            fraction = (target_R - R_1) / (R_2 - R_1)
            Z_intersection = Z_1 + fraction * (Z_2 - Z_1)
            if Z_intersection < 0.0:
                intersections.append(Z_intersection)

    if not intersections:
        raise ValueError(f"The lower wall does not cross R={target_R} m.")

    return np.array([target_R, min(intersections)])


core_boundary_points = eq.separatrix(ntheta=16)
target_strike_radius = 1.5
divertor_control_point = lower_wall_point_at_R(eq, target_strike_radius)
control_points = np.vstack([core_boundary_points, divertor_control_point])

print(f"Number of core control points: {len(core_boundary_points)}")
print(f"Divertor control point [m]: {divertor_control_point}")

In [ ]:
# visualise the fixed core points and the requested divertor point
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 8), dpi=80)

for ax in (ax1, ax2):
    ax.grid(True, which="both", alpha=0.5)
    eq.tokamak.plot(axis=ax, show=False)
    ax.plot(eq.tokamak.wall.R, eq.tokamak.wall.Z, color="k", linewidth=1.2)
    ax.contour(
        eq.R,
        eq.Z,
        eq.psi(),
        levels=[eq.psi_bndry],
        colors="red",
    )
    ax.scatter(
        core_boundary_points[:, 0],
        core_boundary_points[:, 1],
        color="gold",
        edgecolors="k",
        s=35,
        zorder=20,
        label="Core flux constraints",
    )
    ax.scatter(
        *divertor_control_point,
        color="cyan",
        edgecolors="k",
        s=70,
        zorder=20,
        label="Divertor flux target",
    )
    ax.set_aspect("equal")

ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
ax1.legend(loc="upper right")

ax2.set_xlim(0.75, 1.8)
ax2.set_ylim(-2.15, -0.8)
ax2.set_title("Lower divertor")
plt.tight_layout()

### Define the virtual-circuit targets

For this example every target has the same physical meaning and units: the coil contribution to poloidal flux at a fixed spatial point. The target calculator therefore evaluates $\psi_{\mathrm{tokamak}}$ directly through the machine object and returns it in mWb/$2\pi$.

The full Grad-Shafranov problem is still solved for every coil perturbation by the standard virtual-circuit implementation. Although the vacuum-flux response itself is linear in the coil currents, these solves keep the workflow identical to other FreeGSNKE virtual-circuit calculations.

In [ ]:
def psi_tokamak_targets(equilibrium):
    """Evaluate coil flux at the fixed core and divertor control points."""

    psi_tokamak = equilibrium.tokamak.psi(
        control_points[:, 0], control_points[:, 1]
    )
    return 1e3 * np.asarray(psi_tokamak, dtype=float)


target_names = [
    f"psi_core_{index:02d}" for index in range(len(core_boundary_points))
] + ["psi_divertor"]

initial_target_values = psi_tokamak_targets(eq)
print(f"Number of VC targets: {len(target_names)}")
print("Initial divertor coil flux "
      f"= {initial_target_values[-1]:.6f} mWb/2pi")

### Construct the fixed-core virtual circuit

We use the same ten PF circuits selected in Example 09. The resulting shape matrix contains one row per control point and one column per circuit:

$$S_{i,j} = \frac{\partial \psi_{\mathrm{tokamak},i}}{\partial I_j}.$$

There are more rows than columns because the up-down-symmetric circuit definitions make the corresponding upper and lower core constraints mutually consistent.

In [ ]:
from freegsnke import virtual_circuits

VCs = virtual_circuits.VirtualCircuitHandling()
VCs.define_solver(GSStaticSolver, target_relative_tolerance=1e-7)

coils = ["PX", "D1", "D2", "D3", "Dp", "D5", "D6", "D7", "P4", "P5"]

VCs.calculate_VC(
    eq=eq,
    profiles=profiles,
    coils=coils,
    target_names=target_names,
    target_calculator=psi_tokamak_targets,
    starting_dI=None,
    min_starting_dI=50,
    verbose=True,
    name="fixed_core_divertor_VC",
)

In [ ]:
shape_matrix = VCs.fixed_core_divertor_VC.shape_matrix.copy()
matrix_rank = np.linalg.matrix_rank(shape_matrix)
singular_values = np.linalg.svd(shape_matrix, compute_uv=False)
condition_number = np.linalg.cond(shape_matrix)

print(f"Shape matrix dimensions: {shape_matrix.shape}")
print(f"Shape matrix rank: {matrix_rank}")
print(f"Condition number: {condition_number:.3e}")
print(f"Singular values: {singular_values}")

colour_limit = np.max(np.abs(shape_matrix))
fig, ax = plt.subplots(figsize=(9, 6), dpi=80)
image = ax.imshow(
    shape_matrix,
    aspect="auto",
    cmap="coolwarm",
    vmin=-colour_limit,
    vmax=colour_limit,
)
ax.set_xticks(np.arange(len(coils)))
ax.set_xticklabels(coils)
ax.set_yticks(np.arange(len(target_names)))
ax.set_yticklabels(target_names)
ax.set_xlabel("PF circuit")
ax.set_ylabel("Flux target")
ax.set_title(r"Shape matrix $\partial\psi_{tokamak}/\partial I$")
fig.colorbar(image, ax=ax, label="mWb/2pi/A")
plt.tight_layout()

### Request the divertor-field change

The requested shift is zero at every core point. At the divertor point, we calculate the coil-flux change that would make the **initial total flux** equal to the initial separatrix flux:

$$\Delta\psi_{\mathrm{tokamak,div}}
= \psi_{\mathrm{bndry}}
- \psi_{\mathrm{total}}(R_{\mathrm{target}}, Z_{\mathrm{target}}).$$

This uses the initial plasma-flux contribution as a frozen estimate. The subsequent nonlinear forward solve tests whether fixing the core vacuum field is sufficient for that estimate to remain accurate.

In [ ]:
initial_total_psi_at_target = float(
    np.asarray(eq.psiRZ(*divertor_control_point)).item()
)
divertor_flux_shift = 1e3 * (
    eq.psi_bndry - initial_total_psi_at_target
)

requested_target_shifts = np.zeros(len(target_names))
requested_target_shifts[-1] = divertor_flux_shift

print(
    "Requested divertor coil-flux shift "
    f"= {divertor_flux_shift:.6f} mWb/2pi"
)
print(
    "Initial normalized total flux at the target "
    f"= {float(np.asarray(eq.psiNRZ(*divertor_control_point)).item()):.6f}"
)

In [ ]:
# apply the virtual circuit and solve the modified equilibrium
eq_divertor, profiles_divertor, new_target_values, old_target_values = VCs.apply_VC(
    eq=eq,
    profiles=profiles,
    VC_object=VCs.fixed_core_divertor_VC,
    requested_target_shifts=requested_target_shifts,
    verbose=True,
)

actual_target_shifts = new_target_values - old_target_values

### Validate the modified equilibrium

FreeGS4E's strike-point finder reports intersections from every contour at the boundary-flux level, including disconnected contours outside the core. We therefore identify the intended divertor strike point as the outermost wall intersection in each outer quadrant.

In [ ]:
def outer_strikepoint(equilibrium, upper=False):
    """Return the outermost strike point in an upper or lower quadrant."""

    strikepoints = np.atleast_2d(equilibrium.strikepoints()).astype(float)
    axis_R, axis_Z = equilibrium.opt[0, :2]

    if upper:
        mask = (strikepoints[:, 0] > axis_R) & (strikepoints[:, 1] > axis_Z)
    else:
        mask = (strikepoints[:, 0] > axis_R) & (strikepoints[:, 1] < axis_Z)

    candidates = strikepoints[mask]
    if len(candidates) == 0:
        raise RuntimeError("No outer strike point was found in the requested quadrant.")

    return candidates[np.argmax(candidates[:, 0])]


initial_lower_strike = outer_strikepoint(eq, upper=False)
initial_upper_strike = outer_strikepoint(eq, upper=True)
final_lower_strike = outer_strikepoint(eq_divertor, upper=False)
final_upper_strike = outer_strikepoint(eq_divertor, upper=True)

print(f"Initial lower outer strike point [m]: {initial_lower_strike}")
print(f"Final lower outer strike point [m]:   {final_lower_strike}")
print(f"Initial upper outer strike point [m]: {initial_upper_strike}")
print(f"Final upper outer strike point [m]:   {final_upper_strike}")

In [ ]:
# compare vacuum flux over a denser sampling of the original core boundary
dense_core_boundary = eq.separatrix(ntheta=160)
initial_dense_core_flux = eq.tokamak.psi(
    dense_core_boundary[:, 0], dense_core_boundary[:, 1]
)
final_dense_core_flux = eq_divertor.tokamak.psi(
    dense_core_boundary[:, 0], dense_core_boundary[:, 1]
)
dense_core_flux_shift = 1e3 * (
    np.asarray(final_dense_core_flux) - np.asarray(initial_dense_core_flux)
)

# use nearest-neighbour contour distances because separatrix filtering can
# return a slightly different number of points for the two equilibria
final_core_boundary = eq_divertor.separatrix(ntheta=160)
boundary_distances = cdist(dense_core_boundary, final_core_boundary)
initial_to_final = np.min(boundary_distances, axis=1)
final_to_initial = np.min(boundary_distances, axis=0)
boundary_hausdorff = max(
    np.max(initial_to_final), np.max(final_to_initial)
)
boundary_rms_distance = np.sqrt(np.mean(initial_to_final**2))

final_psiN_at_target = float(
    np.asarray(eq_divertor.psiNRZ(*divertor_control_point)).item()
)

print(
    "Maximum sampled-core target residual "
    f"= {np.max(np.abs(actual_target_shifts[:-1])):.3e} mWb/2pi"
)
print(
    "Maximum dense-boundary coil-flux change "
    f"= {np.max(np.abs(dense_core_flux_shift)):.3e} mWb/2pi"
)
print(f"Core-boundary Hausdorff distance = {1e3 * boundary_hausdorff:.3f} mm")
print(f"Core-boundary RMS distance = {1e3 * boundary_rms_distance:.3f} mm")
print(f"Final normalized total flux at target = {final_psiN_at_target:.6f}")

In [ ]:
# calculate and display the PF-current changes used by the virtual circuit
current_shifts = np.array(
    [
        eq_divertor.tokamak[coil].current - eq.tokamak[coil].current
        for coil in coils
    ]
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), dpi=80)

ax1.axhline(0.0, color="k", linewidth=0.8)
ax1.bar(coils, current_shifts, color="royalblue")
ax1.set_xlabel("PF circuit")
ax1.set_ylabel(r"$\Delta I$ [A]")
ax1.set_title("Virtual-circuit current changes")
ax1.grid(True, axis="y", alpha=0.4)

ax2.plot(dense_core_flux_shift, color="darkorange")
ax2.axhline(0.0, color="k", linewidth=0.8)
ax2.set_xlabel("Dense core-boundary sample")
ax2.set_ylabel(r"$\Delta\psi_{tokamak}$ [mWb/2pi]")
ax2.set_title("Vacuum-flux change around the core")
ax2.grid(True, alpha=0.4)

plt.tight_layout()

for coil, shift in zip(coils, current_shifts):
    print(f"{coil:>3s}: {shift:10.1f} A")

In [ ]:
# compare the initial and modified separatrices
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 8), dpi=80)

for ax in (ax1, ax2):
    ax.grid(True, which="both", alpha=0.5)
    eq.tokamak.plot(axis=ax, show=False)
    ax.plot(eq.tokamak.wall.R, eq.tokamak.wall.Z, color="k", linewidth=1.2)
    ax.contour(
        eq.R,
        eq.Z,
        eq.psi(),
        levels=[eq.psi_bndry],
        colors="red",
    )
    ax.contour(
        eq_divertor.R,
        eq_divertor.Z,
        eq_divertor.psi(),
        levels=[eq_divertor.psi_bndry],
        colors="blue",
    )
    ax.scatter(
        core_boundary_points[:, 0],
        core_boundary_points[:, 1],
        color="gold",
        edgecolors="k",
        s=30,
        zorder=20,
    )
    ax.scatter(
        *divertor_control_point,
        color="cyan",
        edgecolors="k",
        s=70,
        zorder=20,
    )
    ax.plot([], [], color="red", label="Initial separatrix")
    ax.plot([], [], color="blue", label="Modified separatrix")
    ax.set_aspect("equal")

ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
ax1.legend(loc="upper right")

ax2.set_xlim(0.75, 1.8)
ax2.set_ylim(-2.15, -0.8)
ax2.set_title("Lower divertor")
plt.tight_layout()

In [ ]:
# concise numerical checks for the demonstration
checks = {
    "full column rank": matrix_rank == len(coils),
    "sampled core flux fixed": np.max(np.abs(actual_target_shifts[:-1])) < 1e-8,
    "dense core flux nearly fixed": np.max(np.abs(dense_core_flux_shift)) < 1e-2,
    "core boundary within 1 mm": boundary_hausdorff < 1e-3,
    "lower strike near R=1.5 m": abs(final_lower_strike[0] - 1.5) < 2e-2,
    "upper strike near R=1.5 m": abs(final_upper_strike[0] - 1.5) < 2e-2,
    "target lies on separatrix": abs(final_psiN_at_target - 1.0) < 1e-3,
}

for description, passed in checks.items():
    print(f"{description:35s}: {passed}")

### Interpretation and limitations

The outer strike points move from approximately $R=1.00$ m to $R=1.50$ m, while the red and blue core boundaries remain visually coincident. The sampled core values of $\psi_{\mathrm{tokamak}}$ are fixed to numerical precision, and the denser validation shows that the field remains almost unchanged between those points as well.

A few qualifications are important:

- The pointwise constraints are an approximation to the spherical-harmonic core constraints used by the Tokamak Exhaust Designer.
- The ten circuits are up-down symmetric in this machine description, so both outer divertor legs move together.
- The sensitivity matrix is ill-conditioned and the large requested displacement produces large current changes, particularly in D2 and D3. Existing virtual-circuit application does not impose current or voltage limits.
- The requested divertor flux shift uses the initial plasma flux as a frozen estimate. This works here because the constrained core field leaves the self-consistent plasma solution almost unchanged; other equilibria or divertor targets may require smaller steps and recalculation of the virtual circuit.
- This example demonstrates numerical capability rather than an operationally feasible MAST-U scenario.